In [ ]:
!pip install biopython

In [ ]:
pip install pyscf ase numpy


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 50.9/50.9 MB 14.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.0/3.0 MB 50.4 MB/s eta 0:00:00


In [ ]:
import numpy as np
from ase.io import read
from pyscf import gto, scf
import os

# 1. Read the molecule from a PDB file
atoms = read('protein_pocket.pdb')

# 2. Build the PySCF Molecule object
mol = gto.Mole()
mol.atom = ""
for atom in atoms:
    mol.atom += f"{atom.symbol} {atom.position[0]} {atom.position[1]} {atom.position[2]}; "
mol.basis = 'sto-3g'
mol.build()

# 3. Run SCF calculation
mf = scf.RHF(mol)
mf.kernel()

# 4. Extract the MO Coefficient matrix
C = mf.mo_coeff  # Shape: (number of basis functions, number of MOs)

# 5. Map basis functions to atoms
ao_labels = mol.ao_labels(fmt=False)  # list of (atom_index, shell_type, ...)

# Prepare output directory
os.makedirs('wavefunctions', exist_ok=True)

# 6. Group basis functions by atoms
atom_basis_indices = {}
for idx, label in enumerate(ao_labels):
    atom_idx = label[0]
    if atom_idx not in atom_basis_indices:
        atom_basis_indices[atom_idx] = []
    atom_basis_indices[atom_idx].append(idx)

# 7. For each atom, save its part of wavefunction
for atom_idx, basis_indices in atom_basis_indices.items():
    atom = atoms[atom_idx]
    atom_label = atom.symbol
    atom_coords = atom.position

    # Slice C matrix rows corresponding to this atom's basis functions
    C_atom = C[basis_indices, :]

    # Bundle coordinates + C_matrix
    data = {
        'coordinates': atom_coords,
        'C_matrix': C_atom
    }

    # Save
    filename = f'wavefunctions/atom_{atom_idx}_{atom_label}.npy'
    np.save(filename, data)

    print(f"Saved wavefunction for atom {atom_idx} ({atom_label}) at '{filename}'")

print("\n All atom wavefunctions saved in 'wavefunctions/' folder.")


converged SCF energy = -1848.48499942649
Saved wavefunction for atom 0 (N) at 'wavefunctions/atom_0_N.npy'
Saved wavefunction for atom 1 (C) at 'wavefunctions/atom_1_C.npy'
Saved wavefunction for atom 2 (C) at 'wavefunctions/atom_2_C.npy'
Saved wavefunction for atom 3 (O) at 'wavefunctions/atom_3_O.npy'
Saved wavefunction for atom 4 (C) at 'wavefunctions/atom_4_C.npy'
Saved wavefunction for atom 5 (C) at 'wavefunctions/atom_5_C.npy'
Saved wavefunction for atom 6 (C) at 'wavefunctions/atom_6_C.npy'
Saved wavefunction for atom 7 (C) at 'wavefunctions/atom_7_C.npy'
Saved wavefunction for atom 8 (C) at 'wavefunctions/atom_8_C.npy'
Saved wavefunction for atom 9 (C) at 'wavefunctions/atom_9_C.npy'
Saved wavefunction for atom 10 (C) at 'wavefunctions/atom_10_C.npy'
Saved wavefunction for atom 11 (O) at 'wavefunctions/atom_11_O.npy'
Saved wavefunction for atom 12 (H) at 'wavefunctions/atom_12_H.npy'
Saved wavefunction for atom 13 (H) at 'wavefunctions/atom_13_H.npy'
Saved wavefunction for atom

In [ ]:
import os
import numpy as np
from pydub import AudioSegment
import scipy.io.wavfile as wav
import math

# 1. Define paths for input (wavefunctions folder) and output (audio folder)
wavefunctions_folder = 'wavefunctions'
output_folder = 'audio_wavefunctions'

# 2. Create the output folder if it doesn't exist
if not os.path.exists(output_folder):
    os.makedirs(output_folder)

# 3. List all .npy files in the wavefunctions folder
files = sorted([f for f in os.listdir(wavefunctions_folder) if f.endswith('.npy')])

# 4. Convert each matrix to audio and save
for file in files:
    file_path = os.path.join(wavefunctions_folder, file)
    data = np.load(file_path, allow_pickle=True).item()

    # Extract the wavefunction matrix
    C_matrix = data['C_matrix']

    # Flatten the matrix into a 1D array for audio generation
    wavefunction_data = C_matrix.flatten()

    # Normalize the wavefunction data to be in the range [-1, 1] (audio format)
    wavefunction_data = wavefunction_data / np.max(np.abs(wavefunction_data))  # Normalize

    # Determine the duration of the audio file (1 minute = 60000 ms)
    target_duration_ms = 60000  # 1 minute = 60000 milliseconds
    sample_rate = 44100  # Sample rate in Hz (standard for audio)

    # Calculate the number of samples needed to play for 1 minute
    num_samples = target_duration_ms * sample_rate // 1000  # Convert to samples

    # If the wavefunction data is shorter than 1 minute, repeat it
    if len(wavefunction_data) < num_samples:
        # Repeat the wavefunction data to match the required number of samples
        num_repeats = math.ceil(num_samples / len(wavefunction_data))
        wavefunction_data = np.tile(wavefunction_data, num_repeats)

    # Trim to the exact number of samples
    wavefunction_data = wavefunction_data[:num_samples]

    # Convert the wavefunction data to audio format (scaled to 16-bit PCM)
    audio_data = np.int16(wavefunction_data * 32767)  # Convert to 16-bit PCM

    # Define the output file name
    output_audio_path = os.path.join(output_folder, f"{file.replace('.npy', '.wav')}")

    # Save the audio data as a .wav file
    wav.write(output_audio_path, sample_rate, audio_data)

    print(f" Saved audio file: {output_audio_path}")

print("\n Finished converting all matrices to audio files.")


In [ ]:
import os
import numpy as np
import scipy.io.wavfile as wav
from pydub import AudioSegment

# 1. Define paths for input (audio_wavefunctions folder) and output (combined audio file)
audio_folder = 'audio_wavefunctions'
output_audio_path = 'combined_wavefunction_audio.wav'

# 2. List all .wav files in the audio folder
audio_files = sorted([f for f in os.listdir(audio_folder) if f.endswith('.wav')])

# 3. Load and combine all audio files
combined_audio = None
sample_rate = 44100  # Standard audio sample rate

for i, audio_file in enumerate(audio_files):
    audio_path = os.path.join(audio_folder, audio_file)
    # Read the audio file
    sample_rate, audio_data = wav.read(audio_path)

    # If combined_audio is None, initialize it with the first audio file
    if combined_audio is None:
        combined_audio = audio_data
    else:
        # Add the current audio's data to the combined signal (interference)
        combined_audio = np.add(combined_audio, audio_data)

    print(f"Adding audio file: {audio_file}")

# 4. Normalize combined audio to avoid clipping (values should be between -32768 and 32767 for 16-bit PCM)
combined_audio = np.clip(combined_audio, -32768, 32767)

# 5. Ensure the audio is exactly 1 minute long
# 1 minute = 60 seconds = 60 * sample_rate samples
target_samples = 60 * sample_rate

if len(combined_audio) < target_samples:
    # If the combined audio is shorter than 1 minute, repeat the signal
    num_repeats = np.ceil(target_samples / len(combined_audio)).astype(int)
    combined_audio = np.tile(combined_audio, num_repeats)[:target_samples]
elif len(combined_audio) > target_samples:
    # Trim to exactly 1 minute if it's longer
    combined_audio = combined_audio[:target_samples]

# 6. Save the combined audio as a .wav file
wav.write(output_audio_path, sample_rate, combined_audio.astype(np.int16))

print(f" Combined audio saved as: {output_audio_path}")


In [ ]:
import numpy as np
import scipy.io.wavfile as wav
import os

# Input and output files
input_audio_path = 'combined_wavefunction_audio.wav'
output_energy_audio_path = 'energy_observable_audio.wav'

# Step 1: Load combined wavefunction audio
sample_rate, audio_data = wav.read(input_audio_path)

# Ensure audio is mono (single channel) for simplicity
if audio_data.ndim > 1:
    audio_data = audio_data.mean(axis=1)

# Step 2: Compute instantaneous energy (square of amplitude)
energy_signal = audio_data.astype(np.float64) ** 2

# Step 3: (Optional) Smooth the energy to make it more playable
# Using a moving average filter
window_size = int(sample_rate * 0.01)  # 10 ms window
energy_signal_smoothed = np.convolve(energy_signal, np.ones(window_size)/window_size, mode='same')

# Step 4: Normalize to 16-bit audio range
energy_signal_smoothed = energy_signal_smoothed / np.max(np.abs(energy_signal_smoothed))  # Normalize to 1
energy_signal_smoothed = (energy_signal_smoothed * 32767).astype(np.int16)  # Scale to int16

# Step 5: Save the energy signal as new wav file
wav.write(output_energy_audio_path, sample_rate, energy_signal_smoothed)

print(f" Energy observable audio saved as: {output_energy_audio_path}")


In [ ]:
import numpy as np
import scipy.io.wavfile as wav
import os

# Input and output
input_audio_path = 'combined_wavefunction_audio.wav'
output_position_audio_path = 'position_observable_audio.wav'

# Step 1: Load combined wavefunction audio
sample_rate, audio_data = wav.read(input_audio_path)

# Ensure mono audio
if audio_data.ndim > 1:
    audio_data = audio_data.mean(axis=1)

# Convert to float
audio_data = audio_data.astype(np.float64)
audio_len = len(audio_data)

# Step 2: Define windowing (simulate position over time)
window_size = int(sample_rate * 0.05)  # 50 ms window
hop_size = int(window_size // 2)

position_values = []

for start in range(0, audio_len - window_size, hop_size):
    window = audio_data[start:start + window_size]
    t = np.arange(len(window))
    prob_density = window**2
    prob_density /= np.sum(prob_density) + 1e-9  # normalize
    expected_position = np.sum(t * prob_density)
    position_values.append(expected_position)

# Step 3: Convert expected positions into an audio waveform
# Normalize to 16-bit range
position_wave = np.interp(np.linspace(0, len(position_values), audio_len),
                          np.arange(len(position_values)),
                          position_values)
position_wave -= np.min(position_wave)
position_wave /= np.max(position_wave)
position_wave = (position_wave * 2 - 1)  # scale to [-1, 1]
position_wave = (position_wave * 32767).astype(np.int16)

# Step 4: Save position waveform
wav.write(output_position_audio_path, sample_rate, position_wave)

print(f" Position observable audio saved as: {output_position_audio_path}")


In [ ]:
import numpy as np
import scipy.io.wavfile as wav
import os

# Input and output paths
input_audio_path = 'combined_wavefunction_audio.wav'
output_audio_path = 'momentum_observable_audio.wav'

# Load combined wavefunction audio
sample_rate, audio_data = wav.read(input_audio_path)

# Normalize audio data
audio_data = audio_data.astype(np.float32)
audio_data /= np.max(np.abs(audio_data)) + 1e-9

# Window size: small slices for local momentum calculation
window_size = int(sample_rate * 0.05)  # 50 ms windows
num_windows = len(audio_data) // window_size

momentum_values = []

# Process each window
for i in range(num_windows):
    start = i * window_size
    end = start + window_size
    window = audio_data[start:end]

    if len(window) == 0:
        continue

    # Perform Fourier Transform (FFT)
    window_fft = np.fft.fft(window)
    window_fft = np.fft.fftshift(window_fft)  # shift zero frequency to center
    freq = np.fft.fftshift(np.fft.fftfreq(len(window), d=1/sample_rate))

    # Probability density in momentum space
    prob_density = np.abs(window_fft)**2
    prob_density /= np.sum(prob_density) + 1e-9  # Normalize

    # Expected momentum (weighted average of frequencies)
    expected_momentum = np.sum(freq * prob_density)

    momentum_values.append(expected_momentum)

# Interpolate back to full length
momentum_wave = np.interp(
    np.linspace(0, len(momentum_values), len(audio_data)),
    np.arange(len(momentum_values)),
    momentum_values
)

# Normalize and scale to int16 range
momentum_wave = (momentum_wave / (np.max(np.abs(momentum_wave)) + 1e-9)) * 32767
momentum_wave = momentum_wave.astype(np.int16)

# Save the momentum observable as a wav file
wav.write(output_audio_path, sample_rate, momentum_wave)

print(f" Extracted momentum observable audio saved as '{output_audio_path}'!")


In [ ]:
import numpy as np
from scipy.io import wavfile

# Observable audio files in the current directory
files = [
    "energy_observable_audio.wav",
    "position_observable_audio.wav",
    "momentum_observable_audio.wav"
]

# Load audio data
audio_data = []
sample_rates = []

for file in files:
    sr, data = wavfile.read(file)
    sample_rates.append(sr)
    audio_data.append(data.astype(np.float32))  # convert to float for safe mixing

# Ensure all sample rates match
if len(set(sample_rates)) != 1:
    raise ValueError("Sample rates of the audio files do not match.")
sample_rate = sample_rates[0]

# Truncate to shortest length
min_len = min([len(data) for data in audio_data])
audio_data = [data[:min_len] for data in audio_data]

# Mix (superpose) all waveforms
combined = np.sum(audio_data, axis=0)

# Normalize to prevent clipping
max_val = np.max(np.abs(combined))
if max_val > 0:
    combined /= max_val

# Convert to 16-bit PCM
combined_int16 = (combined * 32767).astype(np.int16)

# Save the combined file
output_filename = "Sonified.wav"
wavfile.write(output_filename, sample_rate, combined_int16)

print(f" Combined observable audio saved as '{output_filename}'")


In [ ]:
import numpy as np
import scipy.io.wavfile as wav
import os
import math


def generate_cyclohexane_scaffold(audio_path, pdb_path="cyclohexane_scaffold.pdb"):
    """
    Generate a PDB file with carbon-based cyclohexane scaffolds based on audio vibrational patterns.
    The audio data influences the positioning and distortion of the cyclohexane rings.
    Ensures all carbons in each cyclohexane are properly connected.
    """
    # Read audio file
    rate, data = wav.read(audio_path)

    # Use only mono channel if stereo
    if len(data.shape) > 1:
        data = data[:, 0]

    # Normalize audio data
    data = data / np.max(np.abs(data))

    # Calculate number of cyclohexane rings we can generate
    # Each cyclohexane has 6 carbon atoms
    max_rings = 100  # Limit to 100 rings maximum (600 atoms)

    # Downsample to get manageable number of control points for ring positioning
    step = len(data) // max_rings
    if step == 0:
        step = 1
    control_points = data[::step][:max_rings]

    # Get FFT data for additional modulation
    fft_data = np.abs(np.fft.fft(data))
    fft_data = fft_data / np.max(fft_data)
    fft_control = fft_data[::step][:max_rings]

    # Write PDB file
    with open(pdb_path, "w") as f:
        atom_id = 1
        all_connections = []

        for i in range(len(control_points)):
            # Use audio data to position rings and create distortion
            center_x = i * 3.0  # Space rings along x-axis
            center_y = control_points[i] * 10.0  # Y position influenced by audio amplitude
            center_z = fft_control[i] * 5.0  # Z position influenced by frequency content

            # Distortion based on combination of amplitude and frequency
            distortion = (abs(control_points[i]) + fft_control[i]) / 2.0

            # Generate the cyclohexane ring
            ring_coords = create_cyclohexane(center_x, center_y, center_z,
                                             scale=0.8 + fft_control[i] * 0.4,
                                             distortion=distortion)

            # Write atoms for this ring
            ring_first_atom = atom_id
            for x, y, z in ring_coords:
                f.write(f"HETATM{atom_id:5d}  C   CHX A   1    {x:8.3f}{y:8.3f}{z:8.3f}  1.00  0.00           C\n")
                atom_id += 1

            # Store connections for this ring
            for j in range(6):
                atom1 = ring_first_atom + j - 1
                atom2 = ring_first_atom + ((j + 1) % 6) - 1
                all_connections.append((atom1 + 1, atom2 + 1))  # +1 because PDB is 1-indexed

        # Write all connections to ensure each cyclohexane is properly connected
        for atom1, atom2 in all_connections:
            f.write(f"CONECT{atom1:5d}{atom2:5d}\n")

        f.write("END\n")

    print(f"Cyclohexane scaffold PDB file generated: {pdb_path}")
    return pdb_path


def create_cyclohexane(center_x, center_y, center_z, scale=1.0, distortion=0.0):
    """
    Create coordinates for a cyclohexane ring.
    Standard cyclohexane C-C bond length is around 1.5 Å.
    """
    coords = []
    # Standard cyclohexane in chair conformation
    base_coords = [
        (0.0, 1.0, 0.3),  # C1
        (1.0, 0.5, -0.3),  # C2
        (1.0, -0.5, 0.3),  # C3
        (0.0, -1.0, -0.3),  # C4
        (-1.0, -0.5, 0.3),  # C5
        (-1.0, 0.5, -0.3)  # C6
    ]

    # Apply scale and distortion to the ring
    for x, y, z in base_coords:
        # Apply distortion based on audio data
        x_mod = x * (1 + distortion * 0.3)
        y_mod = y * (1 + distortion * 0.2)
        z_mod = z * (1 + distortion * 0.5)

        # Scale and center
        coords.append((
            center_x + x_mod * scale * 1.5,  # 1.5 Å is approximate C-C bond length
            center_y + y_mod * scale * 1.5,
            center_z + z_mod * scale * 1.5
        ))

    return coords


# Generate the cyclohexane scaffold based on the Quantized.wav file
generate_cyclohexane_scaffold("Sonified.wav", "Hit_scaffolds.pdb")


In [ ]:
import os

# Define the path to your audio file
audio_file = 'Sonified.wav'
output_video = 'Sonified.mp4'

# Use ffmpeg to generate a video from the audio waveform
# This command ensures a clear, continuous waveform signal pattern.
os.system(f"ffmpeg -y -i {audio_file} -filter_complex \"showwaves=s=1280x720:mode=default:rate=30\" -t 60 -c:v libx264 -pix_fmt yuv420p {output_video}")

print(f"Video saved as {output_video}")
